# Speed Dating data
Can't know how to clean it


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_rows = 1000
pd.set_option('display.max_columns', None)

In [ ]:
# Load data
data = pd.read_csv('Speed_Dating_Data.csv', sep=',', encoding='latin-1')
data.head(10)

In [ ]:
data.info()

In [ ]:
# 1. BẢNG THÔNG TIN CÁ NHÂN (PERSON_INFO)
# Chứa các đặc điểm nhân khẩu học, học vấn và lối sống (Cố định cho mỗi iid)
# =================================================================
personal_cols = [
    'iid', 'gender', # Định danh và nhóm thử nghiệm
    'age', 'field', 'field_cd', 'undergra',         # Học vấn: Tuổi, ngành học, trường ĐH
    # 'mn_sat', 'tuition', 
    'race', 'imprace',          # Kinh tế/Sắc tộc: Điểm SAT, học phí, tầm quan trọng sắc tộc
    'imprelig', 'from', 'zipcode', 'income',        # Tôn giáo, quê quán, thu nhập gia đình
    'goal', 'date', 'go_out', 'career_c', # Mục tiêu tham gia, tần suất hẹn hò, nghề nghiệp
    'sports', 'tvsports', 'exercise', 'dining',     # Sở thích (1-10): Thể thao, ăn uống...
    'museums', 'art', 'hiking', 'gaming', 'clubbing', 
    'reading', 'tv', 'theater', 'movies', 'concerts', 
    'music', 'shopping', 'yoga',
    # 'exphappy', # Kỳ vọng mức độ hạnh phúc (1-10)
    # 'expnum',   # Dự đoán số người sẽ thích mình (trong 20 người)
    # 'match_es', # Dự đoán số lượng match sẽ có
]



In [ ]:
# Hậu tố _1: Trước sự kiện (Time 1)
time1_cols = [
    'iid', 'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1', # Tìm kiếm ở đối phương
    'attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1', 'shar2_1',         # Nghĩ giới kia tìm gì
    'attr3_1', 'sinc3_1', 'intel3_1', 'fun3_1', 'amb3_1',                   # Tự đánh giá mình
    'attr4_1', 'sinc4_1', 'intel4_1', 'fun4_1', 'amb4_1', 'shar4_1',         # Nghĩ người cùng giới tìm gì
    'attr5_1', 'sinc5_1', 'intel5_1', 'fun5_1', 'amb5_1'                    # Nghĩ người khác đánh giá mình
] 

# Hậu tố _s: Giữa buổi hẹn (Mid-way)
mid_event_cols = [
    'iid', 'attr1_s', 'sinc1_s', 'intel1_s', 'fun1_s', 'amb1_s', 'shar1_s',
    'attr3_s', 'sinc3_s', 'intel3_s', 'fun3_s', 'amb3_s'
] 

# Hậu tố _2: Sau sự kiện 1 ngày (Time 2)
time2_cols = [
    'iid', 'attr1_2', 'sinc1_2', 'intel1_2', 'fun1_2', 'amb1_2', 'shar1_2',
    'attr2_2', 'sinc2_2', 'intel2_2', 'fun2_2', 'amb2_2', 'shar2_2',
    'attr3_2', 'sinc3_2', 'intel3_2', 'fun3_2', 'amb3_2',
    'attr4_2', 'sinc4_2', 'intel4_2', 'fun4_2', 'amb4_2', 'shar4_2',
    'attr5_2', 'sinc5_2', 'intel5_2', 'fun5_2', 'amb5_2',
    'attr7_2', 'sinc7_2', 'intel7_2', 'fun7_2', 'amb7_2', 'shar7_2' # Trọng số thực tế đã dùng để quyết định
] 

# Hậu tố _3: Sau 3-4 tuần (Time 3)
time3_cols = [
    'iid', 'attr1_3', 'sinc1_3', 'intel1_3', 'fun1_3', 'amb1_3', 'shar1_3',
    'attr2_3', 'sinc2_3', 'intel2_3', 'fun2_3', 'amb2_3', 'shar2_3',
    'attr3_3', 'sinc3_3', 'intel3_3', 'fun3_3', 'amb3_3',
    'attr4_3', 'sinc4_3', 'intel4_3', 'fun4_3', 'amb4_3', 'shar4_3',
    'attr5_3', 'sinc5_3', 'intel5_3', 'fun5_3', 'amb5_3',
    'attr7_3', 'sinc7_3', 'intel7_3', 'fun7_3', 'amb7_3', 'shar7_3'
] 
interaction_cols = [
    'iid', 'pid', 'partner', 'match', 'int_corr', 'samerace',
    # Thông tin đối tác (hậu tố _o)
    'age_o', 'race_o', 'pf_o_att', 'pf_o_sin', 'pf_o_int', 'pf_o_fun', 'pf_o_amb', 'pf_o_sha',
    'dec_o', 'attr_o', 'sinc_o', 'intel_o', 'fun_o', 'amb_o', 'shar_o', 'like_o', 'prob_o', 'met_o',
    # Đánh giá của chính người đó về đối tác (Scorecard)
    'dec', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar', 'like', 'prob', 'met'
] 

In [ ]:
data[interaction_cols].head(10)

In [ ]:
results_cols = [
    'iid', 'pid',
    'satis_2', 'length', 'numdat_2', # Phản hồi về sự kiện
    'you_call', 'them_cal', 'date_3', 'numdat_3', 'num_in_3' # Kết quả liên lạc thực tế
] 
data[results_cols][data[results_cols]['numdat_3'].notnull()].head(20)

In [143]:
# Tập hợp tất cả các cột đã phân loại
classified_cols = set(personal_cols + time1_cols + 
                      mid_event_cols + time2_cols + time3_cols + interaction_cols + results_cols)

# Các cột còn lại chưa được đưa vào bảng nào
remaining_cols = [col for col in data.columns if col not in classified_cols]

print(f"Các cột chưa phân loại: {remaining_cols}")

Các cột chưa phân loại: ['id', 'idg', 'condtn', 'wave', 'round', 'position', 'positin1', 'order', 'mn_sat', 'tuition', 'career', 'exphappy', 'expnum', 'match_es']


In [ ]:
data_numeric = data.select_dtypes(include=['number'])
print(f"Dropped columns: {data.columns.difference(data_numeric.columns).tolist()}")

In [ ]:
data['income'] = pd.to_numeric(data['income'].str.replace(',', ''), errors='coerce')


In [ ]:
# Drop non-numeric columns
data_numeric = data.select_dtypes(include=['object'])
print(data_numeric.info())
print(data_numeric.isnull().mean())
# data_numeric.head(50)
# in tỷ lệ null


